# Interactive Notebook 03 - PMSM simulation:

This interactive Jupyter notebook introduces basic simulation for permanent magnet synchronous motors.

For help with the installation of the required software, consider the comments in ```CTPD_course\interactive_notebooks\README.md```.
Throughout the exercises, we will be using a combination of scientific computation libraries from the [JAX](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html) ecosystem and visualize them with [matplotlib](https://matplotlib.org/) and [ipywidgets](https://ipywidgets.readthedocs.io/en/stable/).

### Preliminaries & Imports:

In [ ]:
# automatically reloads imported ```.py```-files once they are changed and saved
%load_ext autoreload
%autoreload 2

In [ ]:
%%html
<style>
div.jupyter-widgets.widget-label {display: none;}
</style>

In [ ]:
# imports required packages
from functools import partial
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
mpl.rcParams.update({'font.size': 20})

import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

In [ ]:
from helper_functions import visualize_eigendynamics, estimate_eigendynamics_grid, visualize_trajectories, aprbs, visualize_flux, visualize_diff_inductance

(**Optional**: If you have LaTeX installed, you can use the following lines for pretty rendering of plot labels.
Any LaTeX installation should work, as long as all the required packages are installed, e.g., [MiKTeX](https://miktex.org/) or [TeXLive](https://www.tug.org/texlive/).

If you do not have LaTeX installed, you can comment the next cell out or skip it.)

In [ ]:
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

---

**Throught the whole notebook, we will assume that the rotation speed $\omega$ can be set arbitrarily from an external load machine.**

### Simulation of the standard linear PMSM with constant inductances:

The standard ODE for the linear PMSM with constant inductances in dq-coordinates (see lecture 3 eq. 3.10) is

$$ \large \frac{\mathrm{d}}{\mathrm{d}t} \boldsymbol{i}_{\mathrm{s, dq}} (t) = \begin{bmatrix} L^{-1}_{\mathrm{dd}, 0} & 0 \\ 0 & L^{-1}_{\mathrm{qq}, 0}  \end{bmatrix} 
\left[ \boldsymbol{u}_{\mathrm{s, dq}} (t) - R_\mathrm{s} \boldsymbol{i}_{\mathrm{s, dq}} (t) - \omega (t) \boldsymbol{J} \left( \begin{bmatrix} 0 \\ \psi_\mathrm{PM} \end{bmatrix} + \begin{bmatrix} L_{\mathrm{dd}, 0} & 0 \\ 0 & L_{\mathrm{qq}, 0}  \end{bmatrix}  \boldsymbol{i}_{\mathrm{s, dq}} (t) \right)  \right] 
$$

To simulate the electrical motor behavior, we first define the necessary parameters as

$$ \large  \boldsymbol{\theta} = \begin{bmatrix} p & R_\mathrm{s} & L_\mathrm{dd} & L_\mathrm{qq} & \psi_\mathrm{PM} \end{bmatrix}.$$


In [ ]:
class LinearParams(eqx.Module):
    p: jax.Array = eqx.field(converter=jnp.asarray)  # Number of pole pairs
    R_s: jax.Array = eqx.field(converter=jnp.asarray)  # Stator resistance
    L_dd: jax.Array = eqx.field(converter=jnp.asarray)  # D-axis inductance
    L_qq: jax.Array = eqx.field(converter=jnp.asarray)  # Q-axis inductance
    psi_pm: jax.Array = eqx.field(converter=jnp.asarray)  # Permanent magnet flux linkage
    i_lim: float
    u_dc: float

linear_motor_params = LinearParams(
    p=3,
    R_s=jnp.array(17.932e-3),
    L_dd=jnp.array(0.37e-3),
    L_qq=jnp.array(1.2e-3),
    psi_pm=jnp.array(65.65e-3),
    i_lim=250,
    u_dc=400,
)

The ODE itself can be implemented as:

In [ ]:
@eqx.filter_jit
def linear_ode(i_dq_t, u_dq_t, omega_t, params):

    L_inv = jnp.array([[1/params.L_dd, 0],[0, 1/params.L_qq]])
    J = jnp.array([[0, -1],[1, 0]])
    psi_pm_vec = jnp.array([params.psi_pm, 0])
    L = jnp.array([[params.L_dd, 0],[0, params.L_qq]])
    
    return L_inv @ (u_dq_t - params.R_s * i_dq_t - omega_t * J @ (psi_pm_vec + L @ i_dq_t))

#### Visualization of the Eigendynamics as a vectorfield:

Without any mechanical rotation and any applied voltage, the currents are driven back to the equilibrium point at $\boldsymbol{i}_\mathrm{dq} = \begin{bmatrix} 0 \, \mathrm{A} & 0 \, \mathrm{A} \end{bmatrix}$.

In [ ]:
fig, axs = visualize_eigendynamics(ode=linear_ode, params=linear_motor_params, n_values=[0])

With higher rotation speed the EMF and the Eigendynamics increase in strength (still no voltages applied):

In [ ]:
fig, axs = visualize_eigendynamics(ode=linear_ode, params=linear_motor_params, n_values=[0, 2500, 5000, 7500, 10000])

#### Explicit Euler:

To simulate trajectories from the model, we require an ODE solver.
As discussed in the lecture, the simplest form is to use explicit Euler

$$ \large \boldsymbol{i}_{\mathrm{s, dq}}[k+1] = \boldsymbol{i}_{\mathrm{s, dq}}[k] + T_\mathrm{s} \boldsymbol{f}_{\boldsymbol{\theta}} (\boldsymbol{i}_{\mathrm{s, dq}}[k], \boldsymbol{u}_{\mathrm{s, dq}}[k], \omega[k]), $$
where 
$$ \large \boldsymbol{f}_{\boldsymbol{\theta}} (\boldsymbol{i}_{\mathrm{s, dq}}[k], \boldsymbol{u}_{\mathrm{s, dq}}[k], \omega[k]) =  \begin{bmatrix} L^{-1}_{\mathrm{dd}, 0} & 0 \\ 0 & L^{-1}_{\mathrm{qq}, 0}  \end{bmatrix} 
\left[ \boldsymbol{u}_{\mathrm{s, dq}}[k] - R_\mathrm{s} \boldsymbol{i}_{\mathrm{s, dq}}[k]- \omega[k] \boldsymbol{J} \left( \begin{bmatrix} 0 \\ \psi_\mathrm{PM} \end{bmatrix} + \begin{bmatrix} L_{\mathrm{dd}, 0} & 0 \\ 0 & L_{\mathrm{qq}, 0}  \end{bmatrix}  \boldsymbol{i}_{\mathrm{s, dq}}[k] \right)  \right] . $$

In [ ]:
def linear_euler_step(i_dq, u_dq, omega, params, T_s):
    return i_dq + T_s * linear_ode(i_dq, u_dq, omega, params)

@eqx.filter_jit
def linear_euler_simulation(i_dq_0, u_dq_sequence, omega, params, T_s):

    def body_function(carry, u_dq):
        i_dq = carry
        i_dq_next = linear_euler_step(i_dq, u_dq, omega, params, T_s)
        return i_dq_next, i_dq_next

    _, i_dq_sequence =  jax.lax.scan(body_function, i_dq_0, u_dq_sequence)
    i_dq_sequence = jnp.concatenate([i_dq_0[None, :], i_dq_sequence], axis=0)
    return i_dq_sequence

In [ ]:
sequence_length = 10_000
T_s = 1e-4
n = 1000
u_dq_sequence = jnp.zeros((sequence_length, 2))

omega = jnp.array(linear_motor_params.p * n * 2 * jnp.pi / 60)

i_dq_sequence = linear_euler_simulation(
    i_dq_0=jnp.array([0.0, 0.0]), u_dq_sequence=u_dq_sequence, omega=omega, params=linear_motor_params, T_s=T_s,
)

visualize_trajectories(
    [i_dq_sequence], [u_dq_sequence], [T_s], [omega], linear_ode, linear_motor_params,
)

Next we consider the behavior of the simulation with different discrete sampling times $T_\mathrm{s} \in [10^{-5}, 10^{-4}, 10^{-3}, 10^{-2}]$:

In [ ]:
i_dq_sequences = []
u_dq_sequences = []
omegas = []

T_s_list = [1e-5, 1e-4, 1e-3, 1e-2]

for T_s in T_s_list:
    i_dq_sequence = linear_euler_simulation(
        i_dq_0=jnp.array([0.0, 0.0]), u_dq_sequence=u_dq_sequence, omega=omega, params=linear_motor_params, T_s=T_s,
    )

    i_dq_sequences.append(i_dq_sequence)
    u_dq_sequences.append(u_dq_sequence)
    omegas.append(omega)

visualize_trajectories(
    i_dq_sequences, u_dq_sequences, T_s_list, omegas, linear_ode, linear_motor_params, labels=[f"$T_\\mathrm{{s}}$={T_s}" for T_s in T_s_list]
)

With small time steps we can get the expected (stable) behavior, but one has to be careful not too choose it to large.
With $T_\mathrm{s} > 10^{-4}$, the system's free response diverges, which is not realistic for the given setup.

Now some voltages $\boldsymbol{u}_\mathrm{dq}$ are applied. It is fairly difficult to keep the PMSM within constraints thorugh open-loop control (which is why we will be taking a loop at closed loop control in the next notebook).

In [ ]:
sequence_length = 10_000
T_s = 1e-4
n = 500
u_dq_sequence = aprbs(sequence_length, 2, t_min=1000, t_max=3000, key=jax.random.key(0)).T[0] * 10

omega = jnp.array(linear_motor_params.p * n * 2 * jnp.pi / 60)

i_dq_sequence = linear_euler_simulation(
    i_dq_0=jnp.array([0.0, 0.0]), u_dq_sequence=u_dq_sequence, omega=omega, params=linear_motor_params, T_s=T_s,
)

visualize_trajectories(
    [i_dq_sequence], [u_dq_sequence], [T_s], [omega], linear_ode, linear_motor_params,
)

#### Advanced ODE solvers:

Next we transfer the ODE into the API of the diffrax Python library. 
This way we can use the advanced ODE solvers that come with the library without having to implement them ourselves.

In [ ]:
def diffrax_linear_ode(t, y, args, action):
    i_dq = y
   
    params, omega = args
    u_dq = action(t)

    return linear_ode(i_dq, u_dq, omega, params)

In [ ]:
@eqx.filter_jit
def linear_diffrax_simulation(i_dq_0, u_dq_sequence, omega, params, T_s, solver, adaptive_stepsize=False):

    def voltage(t):
        return u_dq_sequence[jnp.array(t / T_s, int)]
    
    args = (linear_motor_params, omega)
    vector_field = partial(diffrax_linear_ode, action=voltage)
    
    term = diffrax.ODETerm(vector_field)
    t0 = 0
    t1 = T_s * u_dq_sequence.shape[0]
    
    y0 = i_dq_0
    saveat = diffrax.SaveAt(ts=jnp.linspace(t0, t1, 1 + int(t1 / T_s)))

    if adaptive_stepsize:
        stepsize_controller = diffrax.PIDController(rtol=1e-5, atol=1e-5)
    else:
        stepsize_controller = diffrax.ConstantStepSize()
    
    y = diffrax.diffeqsolve(
        term,
        solver,
        t0,
        t1,
        dt0=T_s,
        y0=y0,
        args=args,
        saveat=saveat,
        stepsize_controller=stepsize_controller,
        max_steps=100_000
    )
    return y.ys


def simulate_response(
    T_s,
    n,
    u_dq,
    sequence_length,
    adaptive_stepsize,
):
    omega = jnp.array(linear_motor_params.p * n * 2 * jnp.pi / 60)
        
    i_dq_sequences = []
    u_dq_sequences = []
    omegas = []

    if adaptive_stepsize:
        solver_list = [diffrax.Tsit5(), diffrax.Tsit5()]
        adaptive_step_size_list = [False, True]
        labels = [r"$\mathrm{Tsit5}_\mathrm{nonadaptive}$", r"$\mathrm{Tsit5}_\mathrm{adaptive}$"]
    else:
        solver_list = [diffrax.Euler(), diffrax.Heun(), diffrax.Tsit5()]
        adaptive_step_size_list = [False, False, False]
        labels=[r"$\mathrm{Euler}$", r"$\mathrm{Heun}$", r"$\mathrm{Tsit5}$"]
    
    for _adaptive_stepsize, solver in zip(adaptive_step_size_list, solver_list):
        i_dq_sequence = linear_diffrax_simulation(
            i_dq_0=jnp.array([0.0, 0.0]),
            u_dq_sequence=u_dq_sequence,
            omega=omega,
            params=linear_motor_params,
            T_s=T_s,
            solver=solver,
            adaptive_stepsize=_adaptive_stepsize,
        )
    
        i_dq_sequences.append(i_dq_sequence)
        u_dq_sequences.append(u_dq_sequence)
        omegas.append(omega)
    
    fig, axs = visualize_trajectories(
        i_dq_sequences,
        u_dq_sequences,
        [T_s for _ in range(len(solver_list))],
        omegas,
        linear_ode,
        linear_motor_params,
        labels=labels,
    )
    return fig, axs, (i_dq_sequences, u_dq_sequences, omegas)

In [ ]:
T_s = 1e-4
n = 1000
sequence_length = 1_000
u_dq_sequence = jnp.zeros((sequence_length, 2))

fig, axs, (i_dq_sequences, u_dq_sequences, omegas) = simulate_response(
    T_s,
    n,
    u_dq_sequence,
    sequence_length,
    adaptive_stepsize=False
);

for sequence in i_dq_sequences:
    print("Final currents: ", sequence[-1, :], "in A")

In [ ]:
T_s = 1e-3
n = 1000
sequence_length = 1_000
u_dq_sequence = jnp.zeros((sequence_length, 2))

fig, axs, (i_dq_sequences, u_dq_sequences, omegas) = simulate_response(
    T_s,
    n,
    u_dq_sequence,
    sequence_length,
    adaptive_stepsize=False
);

for sequence in i_dq_sequences:
    print("Final currents: ", sequence[-1, :], "in A")

Heun and Tsit5 stay stable, while Euler diverges. Heun and Tsit5 require more computational effort though.
In essence, the main advantage of higher order methods comes to play when adaptive stepsizing is used:

In [ ]:
T_s = 1e-2
n = 1000
sequence_length = 1_000
u_dq_sequence = jnp.zeros((sequence_length, 2))

fig, axs, (i_dq_sequences, u_dq_sequences, omegas) = simulate_response(
    T_s,
    n,
    u_dq_sequence,
    sequence_length,
    adaptive_stepsize=True
);


for sequence in i_dq_sequences:
    print("Final currents: ", sequence[-1, :], "in A")

The adaptive stepsizing produces the correct operating points without specifying the necessary sampling time. It automatically selects the necessary steps for the simulation. The result looks odd because the simulation is only reported every $T_\mathrm{s} = 0.01$ and intermediate steps are not saved.

### Nonlinear PMSM model:

First, we load the look-up-tables (LUTs) used for the flux function $\boldsymbol{f}_{\psi_{\mathrm{s}}} (\boldsymbol{i}_{\mathrm{s, dq}}(t))$ and differential inductance matrix $\boldsymbol{L}(\boldsymbol{i}_{\mathrm{s, dq}}(t))$ and visualize them:

In [ ]:
from scipy.io import loadmat
from pathlib import Path

from motor_params import generate_luts, lut_interpolate

In [ ]:
lut_raw = loadmat(Path("LUT_BRUSA_jax_grad.mat"))

In [ ]:
visualize_flux(lut_raw);
visualize_diff_inductance(lut_raw);

To generate values that are not directly part of the LUT, you may simply (linearly) interpolate between the values:

In [ ]:
lut_grids, lut_values = generate_luts(lut_raw)

i_dq = jnp.array([0, 0])
params = {q: lut_interpolate(*lut_grids[q], lut_values[q], *i_dq) for q in lut_grids}
params

#### Simulation with nonlinear ODE:

The nonlinear ODE with respect to the currents is

$$ \large \frac{\mathrm{d}}{\mathrm{d}t} \boldsymbol{i}_{\mathrm{s, dq}} (t) = \boldsymbol{L}^{-1} (\boldsymbol{i}_{\mathrm{s, dq}}(t)) \left[ 
    \boldsymbol{u}_{\mathrm{s, dq}} (t) - R_\mathrm{s} \boldsymbol{i}_{\mathrm{s, dq}} (t) - \omega (t) \boldsymbol{J} \boldsymbol{f}_{\psi_{\mathrm{s}}} (\boldsymbol{i}_{\mathrm{s, dq}}(t))
\right]
$$

In [ ]:
class NonlinearParams(eqx.Module):
    p: jax.Array = eqx.field(converter=jnp.asarray)  # Number of pole pairs
    R_s: jax.Array = eqx.field(converter=jnp.asarray)  # Stator resistance
    lut_grids: dict[str | jax.Array]  # coordinates for measured flux values and differential inductances
    lut_values: dict[str | jax.Array]  # values for measured flux values and differential inductances
    i_lim: float
    u_dc: float

nonlinear_motor_params = NonlinearParams(
    p=3,
    R_s=jnp.array(17.932e-3),
    lut_grids=lut_grids,
    lut_values=lut_values,
    i_lim=250,
    u_dc=400,
)

In [ ]:
@eqx.filter_jit
def nonlinear_ode(i_dq_t, u_dq_t, omega_t, params):

    J = jnp.array([[0, -1],[1, 0]])
    variable_params = {q: lut_interpolate(*params.lut_grids[q], params.lut_values[q], *i_dq_t) for q in params.lut_grids}

    L_diff = jnp.column_stack([variable_params[q] for q in ["L_dd", "L_dq", "L_qd", "L_qq"]]).reshape(2, 2)
    L_diff_inv = jnp.linalg.inv(L_diff)

    psi_dq = jnp.column_stack([variable_params[psi] for psi in ["Psi_d", "Psi_q"]]).reshape(-1)
    
    return L_diff_inv @ (u_dq_t - params.R_s * i_dq_t - omega_t * J @ psi_dq)

In [ ]:
def nonlinear_euler_step(i_dq, u_dq, omega, params, T_s):
    return i_dq + T_s * nonlinear_ode(i_dq, u_dq, omega, params)

@eqx.filter_jit
def nonlinear_euler_simulation(i_dq_0, u_dq_sequence, omega, params, T_s):

    def body_function(carry, u_dq):
        i_dq = carry
        i_dq_next = nonlinear_euler_step(i_dq, u_dq, omega, params, T_s)
        return i_dq_next, i_dq_next

    _, i_dq_sequence =  jax.lax.scan(body_function, i_dq_0, u_dq_sequence)
    i_dq_sequence = jnp.concatenate([i_dq_0[None, :], i_dq_sequence], axis=0)
    return i_dq_sequence

#### Visualize free response:

In [ ]:
fig, axs = visualize_eigendynamics(ode=nonlinear_ode, params=nonlinear_motor_params, n_values=[0, 2500, 5000, 7500, 10000])

#### Visualize simulation:

In [ ]:
sequence_length = 10_000
T_s = 1e-4
n = 400
u_dq_sequence = jnp.zeros((sequence_length, 2))

omega = jnp.array(nonlinear_motor_params.p * n * 2 * jnp.pi / 60)

i_dq_sequence = nonlinear_euler_simulation(
    i_dq_0=jnp.array([0.0, 0.0]), u_dq_sequence=u_dq_sequence, omega=omega, params=nonlinear_motor_params, T_s=T_s,
)

visualize_trajectories(
    [i_dq_sequence], [u_dq_sequence], [T_s], [omega], nonlinear_ode, nonlinear_motor_params,
)

In [ ]:
sequence_length = 10_000
T_s = 1e-4
n = 400
u_dq_sequence = aprbs(sequence_length, 2, t_min=1000, t_max=3000, key=jax.random.key(0)).T[0] * 10

omega = jnp.array(nonlinear_motor_params.p * n * 2 * jnp.pi / 60)

i_dq_sequence = nonlinear_euler_simulation(
    i_dq_0=jnp.array([0.0, 0.0]), u_dq_sequence=u_dq_sequence, omega=omega, params=nonlinear_motor_params, T_s=T_s,
)

visualize_trajectories(
    [i_dq_sequence], [u_dq_sequence], [T_s], [omega], nonlinear_ode, nonlinear_motor_params,
)

Applying some minimal, random voltages already destabilizes the system.
Therefore, we will learn about operating point selection and control concepts.

#### Incorporation of voltage constraints:

Due to the inverter operation including modulation, only voltage targets within the alpha-beta voltage hexagon can be applied.

In [ ]:
@eqx.filter_jit
def constr_nonlinear_ode(i_dq_t, u_dq_t, omega_t, eps_t, params):

    J = jnp.array([[0, -1],[1, 0]])
    variable_params = {q: lut_interpolate(*params.lut_grids[q], params.lut_values[q], *i_dq_t) for q in params.lut_grids}

    L_diff = jnp.column_stack([variable_params[q] for q in ["L_dd", "L_dq", "L_qd", "L_qq"]]).reshape(2, 2)
    L_diff_inv = jnp.linalg.inv(L_diff)

    psi_dq = jnp.column_stack([variable_params[psi] for psi in ["Psi_d", "Psi_q"]]).reshape(-1)
    
    return L_diff_inv @ (u_dq_t - params.R_s * i_dq_t - omega_t * J @ psi_dq)

In [ ]:
import numpy as np

In [ ]:
t32 = jnp.array([[1, 0], [-0.5, 0.5 * jnp.sqrt(3)], [-0.5, -0.5 * jnp.sqrt(3)]])
t23 = 2 / 3 * jnp.array([[1, 0], [-0.5, 0.5 * jnp.sqrt(3)], [-0.5, -0.5 * jnp.sqrt(3)]]).T  # only for abc -> alpha/beta

ROTATION_MAP = np.ones((2, 2, 2), dtype=np.complex64)
ROTATION_MAP[1, 0, 1] = 0.5 * (1 + np.sqrt(3) * 1j)
ROTATION_MAP[1, 1, 0] = 0.5 * (1 - np.sqrt(3) * 1j)
ROTATION_MAP[0, 1, 0] = 0.5 * (-1 - np.sqrt(3) * 1j)
ROTATION_MAP[0, 1, 1] = -1
ROTATION_MAP[0, 0, 1] = 0.5 * (-1 + np.sqrt(3) * 1j)
ROTATION_MAP = jnp.array(ROTATION_MAP)

def t_dq_alpha_beta(eps):
    """Compute the transformation matrix for converting between DQ and Alpha-Beta reference frames."""
    cos = jnp.cos(eps)
    sin = jnp.sin(eps)
    return jnp.column_stack((cos, sin, -sin, cos)).reshape(2, 2)


def dq2abc(u_dq, eps):
    """Transform voltages from DQ coordinates to ABC (three-phase) coordinates."""
    u_abc = t32 @ dq2albet(u_dq, eps).T
    return u_abc.T


def dq2albet(u_dq, eps):
    """Transform voltages from DQ coordinates to Alpha-Beta coordinates."""
    q = t_dq_alpha_beta(-eps)
    u_alpha_beta = q @ u_dq.T

    return u_alpha_beta.T


def albet2dq(u_albet, eps):
    """Transform voltages from Alpha-Beta coordinates to DQ coordinates."""
    q_inv = t_dq_alpha_beta(eps)
    u_dq = q_inv @ u_albet.T

    return u_dq.T


def abc2dq(u_abc, eps):
    """Transform voltages from ABC (three-phase) coordinates to DQ coordinates."""
    u_alpha_beta = t23 @ u_abc.T
    u_dq = albet2dq(u_alpha_beta.T, eps)
    return u_dq


def clip_in_abc_coordinates(u_dq, u_dc, omega_el, eps, tau):
    eps_advanced = step_eps(eps, omega_el, tau, 0.5)
    u_abc = dq2abc(u_dq, eps_advanced)
    # clip in abc coordinates
    u_abc = jnp.clip(u_abc, -u_dc / 2.0, u_dc / 2.0)
    u_dq = abc2dq(u_abc, eps)
    return u_dq


def step_eps(eps, omega_el, tau, tau_scale=1.0):
    eps += omega_el * tau * tau_scale
    eps %= 2 * jnp.pi
    boolean = eps > jnp.pi
    summation_mask = boolean * -2 * jnp.pi
    eps = eps + summation_mask
    return eps


def apply_hex_constraint(u_albet):
    u_albet_c = u_albet[0] + 1j * u_albet[1]
    idx = (jnp.sin(jnp.angle(u_albet_c)[..., jnp.newaxis] - 2 / 3 * jnp.pi * jnp.arange(3)) >= 0).astype(int)
    rot_vec = ROTATION_MAP[idx[0], idx[1], idx[2]]
    # rotate sectors upwards
    u_albet_c = jnp.multiply(u_albet_c, rot_vec)
    u_albet_c = jnp.clip(u_albet_c.real, -2 / 3, 2 / 3) + 1j * u_albet_c.imag
    u_albet_c = u_albet_c.real + 1j * jnp.clip(u_albet_c.imag, 0, 2 / 3 * jnp.sqrt(3))
    u_albet_c = jnp.multiply(u_albet_c, jnp.conjugate(rot_vec))  # rotate back
    return jnp.column_stack([u_albet_c.real, u_albet_c.imag])

In [ ]:
def constr_nonlinear_euler_step(i_dq, u_dq, omega, eps, params, T_s):
    return i_dq + T_s * constr_nonlinear_ode(i_dq, u_dq, omega, eps, params)

@eqx.filter_jit
def constr_nonlinear_euler_simulation(i_dq_0, u_dq_sequence, omega, eps_0, params, T_s):

    def body_function(carry, u_dq):
        (i_dq, eps) = carry

        # u_dq = clip_in_abc_coordinates(u_dq, params.u_dc, omega, eps, T_s)

        ############################################################################
        advanced_angle = step_eps(eps, 1.5, T_s, omega)
        u_albet_norm = dq2albet(
            u_dq / (params.u_dc / 2),
            advanced_angle,
        )
        u_albet_norm = apply_hex_constraint(u_albet_norm)
        u_dq_norm = albet2dq(
            u_albet_norm,
            advanced_angle,
        )
        u_dq = jnp.squeeze(u_dq_norm * (params.u_dc / 2))

        ############################################################################
        
        i_dq_next = constr_nonlinear_euler_step(i_dq, u_dq, omega, eps, params, T_s)

        eps_next = step_eps(eps, omega, T_s)
        return (i_dq_next, eps_next), (i_dq_next, u_dq)

    _, sequences =  jax.lax.scan(body_function, (i_dq_0, eps_0), u_dq_sequence)
    i_dq_sequence, u_dq_sequence = sequences
    
    i_dq_sequence = jnp.concatenate([i_dq_0[None, :], i_dq_sequence], axis=0)
    return i_dq_sequence, u_dq_sequence

In [ ]:
sequence_length = 10_000
T_s = 1e-4
n = 400
u_dq_sequence = aprbs(sequence_length, 2, t_min=1000, t_max=3000, key=jax.random.key(0)).T[0] * 1000

omega = jnp.array(nonlinear_motor_params.p * n * 2 * jnp.pi / 60)

i_dq_sequence, u_dq_sequence = constr_nonlinear_euler_simulation(
    i_dq_0=jnp.array([0.0, 0.0]),
    u_dq_sequence=u_dq_sequence,
    omega=omega,
    eps_0=jnp.array([0.0]),
    params=nonlinear_motor_params,
    T_s=T_s,
)

visualize_trajectories(
    [i_dq_sequence], [u_dq_sequence], [T_s], [omega], nonlinear_ode, nonlinear_motor_params, 
)